# EXP-016 CPU Smoke

A sort / B adjacent / C fallback+density / D UID schema. No train, no submit, no GPU.


In [ ]:
import os
open('sampling_geometry.py','w',encoding='utf-8').write('"""EXP-016 physical geometry / sampling. Isolated from training logic.\n\nLegacy path remains byte-compatible in selection order (UID order preserved).\nPhysical path only changes:\n  1) slice sort key: dot(IPP, slice_normal) with logged fallbacks\n  2) 2.5D triplets: true physical neighbors on the sorted stack\n  3) clip centers: physical-span based (still capped by n_clips)\n\nNo laterality, no MIL, no labels, no loss, no backbone changes.\n"""\nfrom __future__ import annotations\n\nimport os\nfrom collections import Counter\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\n# module-level counters for smoke tests / training logs\nSTATS = {\n    "sort_ipp_z_fallback": 0,\n    "sort_no_iop": 0,\n    "sort_no_ipp": 0,\n    "sort_physical_ok": 0,\n    "triplet_missing_neighbor": 0,\n    "triplet_made": 0,\n    "triplet_gap_relaxed": 0,\n    "triplet_skipped": 0,\n}\n\n\ndef reset_stats() -> None:\n    for k in STATS:\n        STATS[k] = 0\n\n\ndef slice_normal_from_iop(iop) -> np.ndarray | None:\n    """IOP: 6 floats [row_x,row_y,row_z, col_x,col_y,col_z] -> unit normal."""\n    if iop is None:\n        return None\n    try:\n        v = np.asarray(iop, dtype=np.float64).reshape(-1)\n        if v.size < 6:\n            return None\n        row, col = v[:3], v[3:6]\n        n = np.cross(row, col)\n        nn = float(np.linalg.norm(n))\n        if not np.isfinite(nn) or nn < 1e-8:\n            return None\n        return n / nn\n    except Exception:\n        return None\n\n\ndef physical_position(ipp, normal: np.ndarray | None) -> tuple[float | None, str]:\n    """Return (pos, kind) kind in {physical, ipp_z, missing}."""\n    if ipp is None:\n        return None, "missing"\n    try:\n        p = np.asarray(ipp, dtype=np.float64).reshape(-1)\n        if p.size < 3:\n            return None, "missing"\n        if normal is not None:\n            return float(np.dot(p[:3], normal)), "physical"\n        return float(p[2]), "ipp_z"\n    except Exception:\n        return None, "missing"\n\n\ndef sort_slices_physical(files, dcm_reader):\n    """Sort DICOM files by physical stack position.\n\n    dcm_reader(path) -> pydicom dataset or None (header-only ok).\n    Returns (rows, meta) where rows = [(path, pos, kind)], sorted ascending.\n    Fallbacks are counted in STATS and reported in meta.\n    """\n    rows = []\n    normals = []\n    for f in files:\n        ds = None\n        try:\n            ds = dcm_reader(f)\n        except Exception:\n            ds = None\n        iop = ds.get("ImageOrientationPatient") if ds is not None else None\n        ipp = ds.get("ImagePositionPatient") if ds is not None else None\n        n = slice_normal_from_iop(iop)\n        if n is None:\n            STATS["sort_no_iop"] += 1\n        else:\n            normals.append(n)\n        pos, kind = physical_position(ipp, n)\n        if kind == "physical":\n            STATS["sort_physical_ok"] += 1\n        elif kind == "ipp_z":\n            STATS["sort_ipp_z_fallback"] += 1\n        else:\n            STATS["sort_no_ipp"] += 1\n        rows.append((f, pos, kind))\n\n    # if normals disagree a lot, still use per-slice normal (local) — log count\n    meta = {\n        "n": len(files),\n        "n_physical": sum(1 for r in rows if r[2] == "physical"),\n        "n_ipp_z": sum(1 for r in rows if r[2] == "ipp_z"),\n        "n_missing": sum(1 for r in rows if r[2] == "missing"),\n    }\n\n    def key(r):\n        # None last, keep stable among equals\n        pos = r[1]\n        return (pos is None, pos if pos is not None else 0.0, r[0])\n\n    rows.sort(key=key)\n    return rows, meta\n\n\ndef median_spacing(positions) -> float:\n    """Median |Δpos| of consecutive finite positions (stack spacing)."""\n    p = np.asarray(\n        [x for x in positions if x is not None and np.isfinite(x)], dtype=np.float64\n    )\n    if p.size < 2:\n        return float("nan")\n    d = np.abs(np.diff(p))\n    d = d[d > 1e-8]\n    if d.size == 0:\n        return float("nan")\n    return float(np.median(d))\n\n\ndef select_triplet_centers(positions, n_clips: int):\n    """Choose center indices for 2.5D clips using physical span.\n\n    Strategy:\n      - require finite positions for span; fallback to even index centers\n      - centers evenly spaced in *physical position* within [lo, hi]\n      - map each desired position to nearest valid interior index\n      - enforce center in [1, n-2] so neighbors exist\n    Returns list of center indices (length <= n_clips), unique, ascending.\n    """\n    n = len(positions)\n    if n < 3 or n_clips <= 0:\n        return []\n    pos = np.asarray(\n        [np.nan if p is None else float(p) for p in positions], dtype=np.float64\n    )\n    finite = np.isfinite(pos)\n    if finite.sum() < 3:\n        # fallback: even index centers in sorted order\n        step = max(1, (n - 2) // n_clips)\n        return list(range(1, n - 1, step))[:n_clips]\n\n    lo = float(np.nanmin(pos))\n    hi = float(np.nanmax(pos))\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-8:\n        step = max(1, (n - 2) // n_clips)\n        return list(range(1, n - 1, step))[:n_clips]\n\n    # physical targets\n    if n_clips == 1:\n        targets = [0.5 * (lo + hi)]\n    else:\n        targets = [lo + (hi - lo) * (i + 0.5) / n_clips for i in range(n_clips)]\n\n    used = set()\n    centers = []\n    for t in targets:\n        # nearest index among interior slices with finite pos\n        best_i, best_d = None, None\n        for i in range(1, n - 1):\n            if not finite[i]:\n                continue\n            d = abs(pos[i] - t)\n            if best_d is None or d < best_d:\n                best_i, best_d = i, d\n        if best_i is None:\n            continue\n        if best_i in used:\n            # try neighbors\n            for delta in (1, -1, 2, -2):\n                j = best_i + delta\n                if 1 <= j <= n - 2 and j not in used and finite[j]:\n                    best_i = j\n                    break\n            else:\n                continue\n        used.add(best_i)\n        centers.append(best_i)\n    return sorted(centers)\n\n\ndef make_clips_physical(\n    series_rows,\n    n_clips=2,\n    img_size=224,\n    window_fn=None,\n    dcm_read_fn=None,\n    max_gap_factor=3.0,\n    abs_gap_cap=None,\n):\n    """Build 2.5D clips with true physical neighbors.\n\n    series_rows: list of (path, pos, kind) from sort_slices_physical\n                 or legacy (path, z) tuples — we accept both.\n    Returns (clips, clip_meta_list).\n    """\n    assert window_fn is not None and dcm_read_fn is not None\n    # normalize rows to (path, pos)\n    norm = []\n    for r in series_rows:\n        if len(r) == 3:\n            norm.append((r[0], r[1]))\n        else:\n            norm.append((r[0], r[1]))\n    n = len(norm)\n    if n < 3:\n        return [], []\n    paths = [r[0] for r in norm]\n    pos = [r[1] for r in norm]\n    spacing = median_spacing(pos)\n    centers = select_triplet_centers(pos, n_clips)\n    clips, metas = [], []\n    for c in centers:\n        idx_trip = (c - 1, c, c + 1)\n        if idx_trip[0] < 0 or idx_trip[2] >= n:\n            STATS["triplet_skipped"] += 1\n            continue\n        ps = [pos[i] for i in idx_trip]\n        if any(p is None or not np.isfinite(p) for p in ps):\n            STATS["triplet_missing_neighbor"] += 1\n            # still allow if indices are consecutive in sorted order\n            gaps = (None, None)\n            relaxed = True\n        else:\n            gaps = (\n                abs(ps[1] - ps[0]),\n                abs(ps[2] - ps[1]),\n            )\n            max_gap = None\n            if np.isfinite(spacing) and spacing > 0:\n                max_gap = spacing * max_gap_factor\n            if abs_gap_cap is not None:\n                max_gap = min(max_gap, abs_gap_cap) if max_gap else abs_gap_cap\n            if max_gap is not None and (gaps[0] > max_gap or gaps[1] > max_gap):\n                STATS["triplet_gap_relaxed"] += 1\n                relaxed = True\n            else:\n                relaxed = False\n        chan = []\n        ok = True\n        for k in idx_trip:\n            try:\n                dcm = dcm_read_fn(paths[k])\n                chan.append(window_fn(dcm, img_size))\n            except Exception:\n                ok = False\n                break\n        if not ok:\n            STATS["triplet_skipped"] += 1\n            continue\n        clips.append(np.stack(chan, axis=0))\n        STATS["triplet_made"] += 1\n        metas.append(\n            {\n                "center": c,\n                "idx": idx_trip,\n                "pos": ps,\n                "gaps": gaps,\n                "spacing": spacing,\n                "relaxed": relaxed,\n            }\n        )\n    return clips, metas\n')
open('rsna_v3.py','w',encoding='utf-8').write('# -*- coding: utf-8 -*-\n"""\nRSNA Knee Abnormality Detection — v2(按 Anatomical_Plane 分平面独立建模 + 融合)\n================================================================================\n相对 baseline(rsna_baseline.py)的核心改动:\n- baseline 把 study 内所有平面的 series 粗暴拼成通道;v2 按 Anatomical_Plane\n  (Axial / Sagittal / Coronal)各训一个独立 ResNet34,平面内部再做 2.5D clip;\n- test 推理时每个平面模型独立预测,最终按 各平面 OOF 宏平均 AUC 加权融合\n  (权重可 --fuse weight 用 AUC 加权,或 --fuse mean 简单平均);\n- 某 study 缺某平面时,融合只对该 study 实际可用的平面归一化权重。\n\n其余逻辑(软标签填充、GroupKFold、valid 掩码损失、DICOM 预处理)与 baseline 一致。\n\n运行环境:Kaggle Notebook(GPU,挂载比赛数据 + 社区软标签数据集)。\n用法:\n    !python rsna_v2.py --epochs 3 --batch 16 --n-clips 2 --planes sagittal,coronal\n    !python rsna_v2.py --epochs 3 --batch 16 --fuse weight --pseudo-conf 0.3\n输出: submission.csv(StudyInstanceUID + 12 列置信分数)\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\ntry:\n    import torch\n    import torch.nn as nn\n    from torch.utils.data import Dataset, DataLoader\n    from torch.optim import AdamW\n    from torch.optim.lr_scheduler import CosineAnnealingLR\n    HAS_TORCH = True\nexcept ImportError:\n    HAS_TORCH = False\n\ntry:\n    import torchvision.models as tvm\n    HAS_TV = True\nexcept ImportError:\n    HAS_TV = False\n\ntry:\n    import pydicom\n    HAS_PYDICOM = True\nexcept ImportError:\n    HAS_PYDICOM = False\n\ntry:\n    import rsna_report_labels as RL   # 报告→12 病灶弱标签(与 rsna_v3.py 同目录)\n    HAS_RL = True\nexcept ImportError:\n    HAS_RL = False\n\ntry:\n    import sampling_geometry as SG  # EXP-016 physical sort / 2.5D sampling\n    HAS_SG = True\nexcept ImportError:\n    HAS_SG = False\n\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.metrics import roc_auc_score\n\nwarnings.filterwarnings("ignore")\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",\n          "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis",\n          "Baker\'s", "Contusion", "Fracture"]\nLABELS_EN = ["ACL", "MCL", "MedialMeniscus", "LateralMeniscus",\n             "MedialOA", "LateralOA", "PFOA", "Effusion", "Synovitis",\n             "Bakers", "Contusion", "Fracture"]  # 列名安全化(写提交用)\nPLANES = ["Axial", "Sagittal", "Coronal"]\n\nIMG_SIZE = 224\nCLIP = 3          # 每 clip 切片数(prev/center/next)\nMAX_SERIES = 4    # 每 study 每个平面最多取的 series 数(v2 按平面取,单平面 series 更少,4 足够)\nFOLDS = 5\nSEED = 42\n\n\ndef resolve_paths():\n    """数据定位顺序:1) /kaggle/input 挂载;2) kagglehub 自动下载;3) 本地 rsna_knee/ 目录。"""\n    # 1) Kaggle 挂载(兼容多层挂载:/kaggle/input/<数据名>/ 或 /kaggle/input/<分类>/<数据名>/)\n    if os.path.isdir("/kaggle/input"):\n        for sub in sorted(os.listdir("/kaggle/input")):\n            d = os.path.join("/kaggle/input", sub)\n            for cand in (d, os.path.join(d, sub)):  # 处理 Kaggle 双层挂载\n                if os.path.exists(os.path.join(cand, "train.csv")):\n                    return cand, "/kaggle/working"\n            if os.path.isdir(d):  # 再往下一层找(如 /kaggle/input/competitions/<name>/)\n                for sub2 in sorted(os.listdir(d)):\n                    cand = os.path.join(d, sub2)\n                    if os.path.exists(os.path.join(cand, "train.csv")):\n                        return cand, "/kaggle/working"\n        print("[诊断] /kaggle/input 存在,但没找到含 train.csv 的目录。实际挂载内容如下:", flush=True)\n        for sub in sorted(os.listdir("/kaggle/input")):\n            p = os.path.join("/kaggle/input", sub)\n            try:\n                inner = os.listdir(p)\n            except OSError:\n                inner = ["(无法读取)"]\n            print(f"  /kaggle/input/{sub}/  ->  {inner[:8]}", flush=True)\n        print("[诊断] 如果你挂的是 \'作者数据包/别人的 kernel 输出\', 里面没有 train.csv 和 train_series/,",\n              "必须挂比赛数据本体 rsna-knee-abnormality-detection (Add Input -> Competitions 标签页)", flush=True)\n    # 2) kagglehub 兜底:未挂载时自动下载比赛数据(Kaggle Notebook 内置凭据)\n    try:\n        import kagglehub\n        d = kagglehub.competition_download("rsna-knee-abnormality-detection")\n        if os.path.exists(os.path.join(d, "train.csv")):\n            print(f"[路径] 未挂载,已用 kagglehub 获取数据: {d}", flush=True)\n            return d, "/kaggle/working"\n    except ImportError:\n        pass  # 本地未装 kagglehub\n    except Exception as e:\n        print(f"[提示] kagglehub 下载数据失败: {e}", flush=True)\n    # 3) 本地回退(脚本目录 / cwd / cwd 下的 rsna_knee 子目录都找)\n    cands = []\n    try:\n        cands.append(os.path.dirname(os.path.abspath(__file__)))\n    except NameError:\n        pass\n    cands.append(os.getcwd())\n    for base in list(cands):\n        cands.append(os.path.join(base, "rsna_knee"))\n    seen = set()\n    for base in cands:\n        if base in seen:\n            continue\n        seen.add(base)\n        if os.path.exists(os.path.join(base, "train.csv")):\n            return base, base\n    sys.exit("找不到数据:请先在比赛页加入比赛并同意数据规则,再 Add Input 挂载 "\n             "rsna-knee-abnormality-detection;本地请在 rsna_knee/ 下运行。")\n\n\n_PSEUDO_FILES = ("train_folds_with_pseudo.csv", "train_with_pseudo.csv",\n                 "train_with_soft_labels.csv", "train_pseudo.csv")\n_SKIP_DIRS = {"train_series", "test_series"}  # DICOM 大目录,绝不去遍历\n_WEIGHT_EXT = (".pth", ".pt", ".ckpt", ".pth.tar")  # 离线预训练权重文件扩展名\n\n# 12 标签多语言关键词词典(英/西/荷/德/法)——零挂载软标签兜底\n# 命中 → 高置信,未命中 → 低置信(软标签,不给硬 0/1,避免错词误导)\n_REPORT_KEYWORDS = {\n    "ACL": ["acl", "anterior cruciate", "cruciado anterior", "voorste kruisband",\n            "vorderen kreuzband", "lca", "ligament croisé antérieur"],\n    "MCL": ["mcl", "medial collateral", "colateral medial", "mediale collaterale",\n            "inneres seitenband", "ligament collatéral médial"],\n    "Medial Meniscus": ["medial meniscus", "menisco interno", "meniscus medialis",\n                        "meniskus medial", "menisque médial", "inner meniscus", "menisco medial"],\n    "Lateral Meniscus": ["lateral meniscus", "menisco externo", "meniscus lateralis",\n                         "meniskus lateral", "menisque latéral", "outer meniscus"],\n    "Medial OA": ["medial osteoarthritis", "artrosis femorotibial medial",\n                  "mediale gonartrose", "mediale arthrose", "osteoartritis medial",\n                  "medial compartment narrowing", "medial joint space narrowing"],\n    "Lateral OA": ["lateral osteoarthritis", "artrosis femorotibial lateral",\n                   "laterale gonartrose", "laterale arthrose", "lateral joint space narrowing"],\n    "PF OA": ["patellofemoral osteoarthritis", "artrosis patelofemoral",\n              "patellofemorale artrose", "patellofemoral arthrose", "pf oa",\n              "patellofemoral narrowing", "patellofemoral joint space"],\n    "Effusion": ["effusion", "derrame", "vocht", "erguss", "épanchement", "joint effusion"],\n    "Synovitis": ["synovitis", "sinovitis", "synoviale", "synovial thickening", "synovitis"],\n    "Baker\'s": ["baker", "popliteal cyst", "quiste de baker", "baker cyst"],\n    "Contusion": ["contusion", "bone bruise", "contusión", "kneuzing", "kontusion",\n                  "marrow edema", "óseo", "bone bruising"],\n    "Fracture": ["fracture", "fractura", "fractuur", "fraktur", "fracture line",\n                 "fracture line", "subchondral fracture"],\n}\n\n\ndef _pseudo_from_reports(train_csv, conf_hit=0.9, conf_miss=0.1):\n    """从放射报告文本用多语言关键词生成 12 标签软标签(零挂载兜底)。\n\n    返回 DataFrame(StudyInstanceUID + pseudo_* 列)。关键词命中→conf_hit,\n    未命中→conf_miss(不给硬 0/1,降低错词误导)。官方已有标注的 study\n    仍以官方为准(load_labels 的填充逻辑保证官方优先)。\n    """\n    if "Report" not in train_csv.columns:\n        return None\n    rep = train_csv["Report"].fillna("")\n    rows = {}\n    for lab, kws in _REPORT_KEYWORDS.items():\n        low = [k.lower() for k in kws]\n        rows[f"pseudo_{lab}"] = rep.apply(\n            lambda t: conf_hit if any(k in str(t).lower() for k in low) else conf_miss)\n    df = pd.DataFrame({"StudyInstanceUID": train_csv["StudyInstanceUID"], **rows})\n    return df\n\n\ndef _print_mount_tree(base="/kaggle/input", tag="数据", max_files=6):\n    """打印挂载目录树(最多两层,每目录前几个文件),帮用户确认挂载内容。"""\n    if not os.path.isdir(base):\n        print(f"[诊断-{tag}] 目录不存在: {base}", flush=True)\n        return\n    print(f"[诊断-{tag}] /kaggle/input 实际挂载内容:", flush=True)\n    for sub in sorted(os.listdir(base)):\n        p = os.path.join(base, sub)\n        try:\n            if os.path.isdir(p):\n                inner = sorted(os.listdir(p))[:max_files]\n                print(f"  {sub}/ -> {inner}", flush=True)\n            else:\n                print(f"  {sub} (文件)", flush=True)\n        except OSError:\n            print(f"  {sub} (无法读取)", flush=True)\n\n\ndef _print_dcm_tree(root, tag="DICOM", max_show=5):\n    """打印 train_series 目录前几层真实结构 + 抽样 .dcm 计数,确认 DICOM 布局。\n\n    RSNA 常见布局:train_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm(两层);\n    也兼容一层(<SeriesInstanceUID>/*.dcm)或更深层,这里打印实际看到的。\n    """\n    dcm = os.path.join(root, "train_series")\n    if not os.path.isdir(dcm):\n        print(f"[诊断-{tag}] {dcm} 不存在!", flush=True)\n        return\n    try:\n        subs_all = sorted(os.listdir(dcm))\n    except OSError:\n        subs_all = []\n    subs = subs_all[:max_show]\n    print(f"[诊断-{tag}] {dcm}", flush=True)\n    print(f"[诊断-{tag}] 子目录/文件数: {len(subs_all)},样本:", flush=True)\n    for s in subs:\n        sp = os.path.join(dcm, s)\n        if os.path.isdir(sp):\n            try:\n                inner = sorted(os.listdir(sp))[:max_show]\n            except OSError:\n                inner = []\n            print(f"  {s}/ -> {inner}", flush=True)\n            for it in inner[:2]:\n                itp = os.path.join(sp, it)\n                if os.path.isdir(itp):\n                    try:\n                        files = sorted(os.listdir(itp))[:3]\n                    except OSError:\n                        files = []\n                    print(f"      {it}/ -> {files}", flush=True)\n        else:\n            print(f"  {s} (文件)", flush=True)\n    # 抽样 .dcm 计数:前 3 study × 前 3 series\n    n_dcm = 0\n    try:\n        for s in subs_all[:3]:\n            sp = os.path.join(dcm, s)\n            if not os.path.isdir(sp):\n                continue\n            for it in sorted(os.listdir(sp))[:3]:\n                itp = os.path.join(sp, it)\n                if os.path.isdir(itp):\n                    n_dcm += sum(1 for f in os.listdir(itp) if f.lower().endswith(".dcm"))\n    except OSError:\n        pass\n    print(f"[诊断-{tag}] 抽样 .dcm 文件数: {n_dcm}(前 3 study × 前 3 series)", flush=True)\n\n\ndef _scan_offline_weights(base="/kaggle/input", depth=0, max_depth=4):\n    """递归扫描挂载目录下的预训练权重文件(.pth/.pt 等),跳过 DICOM 大目录。\n\n    返回 [(绝对路径, 相对 /kaggle/input 的描述)];找不到返回空列表。\n    社区打包的离线权重数据集(如 matteonaccarato/resnet34-imagenet-pth、\n    marcshade/resnet-pretrained-weights-imagenet、pytorch/resnet34)挂载后即可被扫到,\n    无需联网。\n    """\n    hits = []\n    if depth > max_depth or not os.path.isdir(base):\n        return hits\n    try:\n        entries = sorted(os.listdir(base))\n    except OSError:\n        return hits\n    for fn in entries:\n        p = os.path.join(base, fn)\n        if os.path.isfile(p) and fn.lower().endswith(_WEIGHT_EXT):\n            hits.append((p, os.path.relpath(p, "/kaggle/input")))\n        elif os.path.isdir(p) and fn not in _SKIP_DIRS:\n            hits.extend(_scan_offline_weights(p, depth + 1, max_depth))\n    return hits\n\n\ndef _load_state_dict_from_file(path):\n    """从权重文件读 state_dict(兼容:直接 state_dict / 包在 checkpoint 里 / 各种前缀)。\n\n    读取失败或格式不对返回 None。\n    """\n    try:\n        obj = torch.load(path, map_location="cpu", weights_only=True)\n    except TypeError:  # 旧版 torch 无 weights_only 参数\n        obj = torch.load(path, map_location="cpu")\n    except Exception as e:\n        print(f"[权重] {path} 读取失败: {e}", flush=True)\n        return None\n    if isinstance(obj, dict) and "state_dict" in obj:\n        obj = obj["state_dict"]\n    elif isinstance(obj, dict) and "model_state_dict" in obj:\n        obj = obj["model_state_dict"]\n    elif isinstance(obj, dict) and "model" in obj and isinstance(obj["model"], dict):\n        obj = obj["model"]\n    if not isinstance(obj, dict):\n        return None\n    # 剥常见前缀:module. / model. / backbone.(若整体带前缀,统一还原)\n    out = {}\n    for k, v in obj.items():\n        for pre in ("module.", "model.", "backbone."):\n            if k.startswith(pre):\n                k = k[len(pre):]\n                break\n        out[k] = v\n    return out\n\n\ndef _try_load_offline(net, paths):\n    """按序尝试把候选离线权重加载进 net(state_dict 结构须与 ResNet34 匹配)。\n\n    适配规则(面向社区 RSNA 权重,如 barun2104 的 best_resnet34_fold0.pth):\n    - conv1.weight 通道数不同时:reshape+mean 适配到 net 的 conv1 通道数(如 24→3,\n      barun2104 是 8 clip × 3 通道的 2.5D 模型);\n    - fc.* / head.* / classifier.* 一律剥离(我们的模型会替换成 12 头);\n    返回 (是否成功, 命中描述)。\n    """\n    for p, desc in paths:\n        sd = _load_state_dict_from_file(p)\n        if sd is None:\n            continue\n        # 1) 适配 conv1:通道数不同则 reshape+mean(如 24 通道 → 3 通道)\n        if "conv1.weight" in sd:\n            w = sd["conv1.weight"]\n            target = net.conv1.weight.shape[1]\n            if w.shape[1] != target and w.shape[1] % target == 0:\n                B, C, H, W = w.shape\n                w2 = w.reshape(B, target, C // target, H, W).mean(dim=2)\n                sd["conv1.weight"] = w2\n                print(f"[权重] {desc} conv1 通道 {C}→{target} 平均适配", flush=True)\n            elif w.shape[1] != target:\n                sd.pop("conv1.weight")  # 无法适配,丢弃(用随机初始化)\n        # 2) 剥离 fc/head/classifier(模型会替换成 12 头)\n        for k in list(sd.keys()):\n            if k.startswith(("fc.", "head.", "classifier.")):\n                sd.pop(k)\n        try:\n            missing, unexpected = net.load_state_dict(sd, strict=False)\n        except Exception as e:\n            print(f"[权重] {desc} 与 ResNet34 结构不匹配: {e}", flush=True)\n            continue\n        exempt = {k for k in missing if k.startswith(("conv1.", "fc."))}\n        if not unexpected and len(missing) == len(exempt):\n            print(f"[权重] 离线预训练加载成功: {desc} (覆盖 {len(sd)} 个参数张量)", flush=True)\n            return True, desc\n        print(f"[权重] {desc} 结构不匹配(missing {len(missing) - len(exempt)} 个非豁免 key, "\n              f"unexpected {len(unexpected)}),尝试下一个", flush=True)\n    return False, ""\n\n\ndef _scan_pseudo(base, depth=0, max_depth=3):\n    """递归在挂载目录下找软标签 CSV(限制深度,跳过 DICOM 目录)。\n\n    返回 (DataFrame 或 None, 来源描述)。识别规则:含 StudyInstanceUID 且至少有一个\n    pseudo_ 前缀列(如 pseudo_ACL)。找不到返回 (None, "")。\n\n    识别方式双通道:\n    1. 文件名白名单(_PSEUDO_FILES)快速命中;\n    2. **按内容识别**:任意 *.csv 只要表头含 StudyInstanceUID + pseudo_* 列即命中\n       (不依赖文件名,社区数据集改名/换结构也能扫到)。\n    """\n    if depth > max_depth or not os.path.isdir(base):\n        return None, ""\n    try:\n        entries = sorted(os.listdir(base))\n    except OSError:\n        return None, ""\n    # 通道 1:文件名白名单(向后兼容,读全量)\n    for fn in _PSEUDO_FILES:\n        p = os.path.join(base, fn)\n        if os.path.exists(p) and os.path.isfile(p):\n            try:\n                cand = pd.read_csv(p)\n            except Exception:\n                continue\n            if "StudyInstanceUID" in cand.columns and any(\n                    c.startswith("pseudo_") for c in cand.columns):\n                return cand, os.path.relpath(p, "/kaggle/input")\n    # 通道 2:按内容识别(只读表头 nrows=3 判断,命中再全读)\n    for fn in entries:\n        if not fn.lower().endswith(".csv"):\n            continue\n        p = os.path.join(base, fn)\n        if not os.path.isfile(p):\n            continue\n        try:\n            head = pd.read_csv(p, nrows=3)\n        except Exception:\n            continue\n        if "StudyInstanceUID" in head.columns and any(\n                c.startswith("pseudo_") for c in head.columns):\n            try:\n                cand = pd.read_csv(p)\n                return cand, os.path.relpath(p, "/kaggle/input")\n            except Exception:\n                continue\n    for sub in entries:\n        if sub in _SKIP_DIRS:\n            continue\n        cand, src = _scan_pseudo(os.path.join(base, sub), depth + 1, max_depth)\n        if cand is not None:\n            return cand, src\n    return None, ""\n\n\ndef find_pseudo_dataset():\n    """在 /kaggle/input 下递归扫描社区软标签数据集(LLM 从报告提取的伪标签)。\n\n    兼容单层(/kaggle/input/<slug>/)与多层(/kaggle/input/datasets/<slug>/)挂载。\n    返回 (DataFrame 或 None, 来源描述);找不到返回 (None, "")。\n    """\n    if not os.path.isdir("/kaggle/input"):\n        return None, ""\n    return _scan_pseudo("/kaggle/input")\n\n\ndef load_labels(root, pseudo_conf=0.0, use_report_labels=True):\n    """返回 train study 级标签 DataFrame(行=StudyInstanceUID, 列=LABELS, 分数/NaN)。\n\n    标签来源优先级(2026-09-06 起;旧的"社区软标签数据集"方案已证伪弃用——\n    barun2104 的 pseudo_* 98.7% 是字面 0.5 死值、零增量信息,是 loss 恒 0.693\n    的真根因):\n    1. 官方 train.csv 标注(0/1,仅 58/4407 例);\n    2. 内置报告规则抽取器 rsna_report_labels(移植自 rsna-knee-baseline-v1,\n       多语言/按句否定极性/严重度分级):对每份放射报告产出 12 病灶 soft 分数\n       (阳性 0.72~0.95 / 阴性 0.04~0.20 / 无证据=NaN);\n       官方缺失格用抽取分数填充,无证据格保持 NaN(不进损失,由 valid 掩码排除)。\n    """\n    train = pd.read_csv(os.path.join(root, "train.csv"))\n    df = train[["StudyInstanceUID"] + LABELS].copy()\n    official_cnt = {c: int(df[c].notna().sum()) for c in LABELS}\n\n    # EXP-013: multi-source consensus soft labels\n    if os.environ.get("EXP013_MULTISOURCE", ""):\n        from exp013_multisource import build_exp013_labels\n        out = build_exp013_labels(os.path.join(root, "train.csv"), verbose=True)\n        print("[标签] EXP-013 multi-source table loaded")\n        return out\n    # EXP-012: LLM-primary soft labels (gold wins; silence 0.25/0.5 → NaN)\n    exp012 = os.environ.get("EXP012_LLM_PRIMARY", "")\n    if exp012:\n        from exp012_llm_primary import build_exp012_labels\n        out = build_exp012_labels(os.path.join(root, "train.csv"), verbose=True)\n        print("[标签] EXP-012 LLM-primary table loaded")\n        return out\n\n    if not use_report_labels:\n        print("[标签] --no-report-labels:仅官方标注可用")\n        return df\n    if not HAS_RL:\n        print("[标签] 未找到 rsna_report_labels.py(需与 rsna_v3.py 同目录),"\n              "仅官方标注可用")\n        return df\n    if "Report" not in train.columns:\n        print("[标签] train.csv 无 Report 列,仅官方标注可用")\n        return df\n\n    pseudo = RL.build_pseudo_table(train, verbose=True)\n    pseudo = pseudo.set_index("StudyInstanceUID")\n    print(f"[标签] 填充统计({len(df)} 个 study):")\n    for c in LABELS:\n        mask = df[c].isna().values\n        if not mask.any():\n            print(f"  {c:18s} 官方全量,无需填充")\n            continue\n        uids = df.loc[mask, "StudyInstanceUID"]\n        vals = pseudo["pseudo_" + c].reindex(uids).values\n        n_fill = int(pd.notna(vals).sum())\n        df.loc[mask, c] = vals\n        print(f"  {c:18s} 官方 {official_cnt[c]:4d} + 报告抽取填充 {n_fill:4d}"\n              f" = 有效 {int(df[c].notna().sum()):5d}")\n    return df\n\n\n_DCM_MISS_PRINTED = [False]  # 只打印一次 DICOM 未命中诊断,避免 2 万 series 刷屏\n\n\ndef read_dicom_slices(dcm_dir, series_uid, study_uid=None, sampling_mode="legacy"):\n    """读取一个 series 的所有切片并排序,返回 [(path, pos)]。\n\n    sampling_mode:\n      legacy  — 按 ImagePositionPatient[2]（历史行为, EXP-00x baseline）\n      physical — 按 dot(ImagePositionPatient, slice_normal)（EXP-016）\n    """\n    cands = [os.path.join(dcm_dir, series_uid)]\n    if study_uid:\n        cands.append(os.path.join(dcm_dir, study_uid, series_uid))\n    sdir = next((c for c in cands if os.path.isdir(c)), None)\n    if sdir is None:\n        if not _DCM_MISS_PRINTED[0]:\n            _DCM_MISS_PRINTED[0] = True\n            print(f"[诊断-DICOM] 找不到 series 目录,尝试过: {cands}", flush=True)\n            print(f"[诊断-DICOM] {dcm_dir} 是否存在: {os.path.isdir(dcm_dir)}", flush=True)\n            try:\n                inner = sorted(os.listdir(dcm_dir))[:8] if os.path.isdir(dcm_dir) else []\n                print(f"[诊断-DICOM] {dcm_dir} 实际内容样本: {inner}", flush=True)\n            except OSError:\n                pass\n        return []\n    files = [os.path.join(sdir, f) for f in os.listdir(sdir) if f.lower().endswith(".dcm")]\n    if not files:\n        if not _DCM_MISS_PRINTED[0]:\n            _DCM_MISS_PRINTED[0] = True\n            print(f"[诊断-DICOM] 目录存在但无 .dcm 文件: {sdir}", flush=True)\n            print(f"[诊断-DICOM] 目录内容样本: {sorted(os.listdir(sdir))[:8]}", flush=True)\n        return []\n    if sampling_mode == "physical" and HAS_SG:\n        def _hdr(path):\n            return pydicom.dcmread(path, stop_before_pixels=True)\n\n        rows3, meta = SG.sort_slices_physical(files, _hdr)\n        # return (path, pos) compatible with legacy consumers\n        return [(f, p) for (f, p, _k) in rows3]\n    rows = []\n    for f in files:\n        try:\n            d = pydicom.dcmread(f, stop_before_pixels=False)\n            pos = d.get("ImagePositionPatient")\n            z = float(pos[2]) if pos is not None and len(pos) >= 3 else None\n            rows.append((f, z))\n        except Exception:\n            continue\n    rows.sort(key=lambda r: (r[1] is None, r[1] if r[1] is not None else 0.0))\n    return rows\n\n\n# ---------------- DICOM -> 2.5D clip 预处理 ----------------\n\ndef window_and_resize(dcm, img_size=IMG_SIZE):\n    """单张切片:Rescale 后百分位窗口化 -> [0,1] -> resize 224。"""\n    arr = dcm.pixel_array.astype(np.float32)\n    slope = float(dcm.get("RescaleSlope", 1))\n    inter = float(dcm.get("RescaleIntercept", 0))\n    arr = arr * slope + inter\n    if dcm.get("PhotometricInterpretation") == "MONOCHROME1":\n        arr = arr.max() - arr\n    lo, hi = np.percentile(arr, [1, 99])\n    arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)\n    from PIL import Image\n    img = Image.fromarray((arr * 255).astype(np.uint8)).resize((img_size, img_size), Image.BILINEAR)\n    return np.asarray(img, dtype=np.float32) / 255.0\n\n\ndef make_clips(series_rows, n_clips=2, img_size=IMG_SIZE, sampling_mode="legacy"):\n    """从排序好的切片列表生成若干 3 通道 clip。\n\n    legacy  : 均匀 index 步进,三元组 = (c-1,c,c+1) 在排序数组上的邻居\n    physical: 按物理跨度选 center,三元组必须在排序后的物理位置上真正相邻\n              (gap 相对 median spacing 判定,见 sampling_geometry)\n    """\n    if sampling_mode == "physical" and HAS_SG:\n        clips, _metas = SG.make_clips_physical(\n            series_rows,\n            n_clips=n_clips,\n            img_size=img_size,\n            window_fn=window_and_resize,\n            dcm_read_fn=lambda p: pydicom.dcmread(p),\n        )\n        return clips\n    clips = []\n    paths = [r[0] for r in series_rows]\n    if len(paths) < CLIP:\n        return clips\n    step = max(1, (len(paths) - 2) // n_clips)\n    for c in range(1, len(paths) - 1, step):\n        if len(clips) >= n_clips:\n            break\n        chan = []\n        ok = True\n        for k in (c - 1, c, c + 1):\n            try:\n                dcm = pydicom.dcmread(paths[k])\n                chan.append(window_and_resize(dcm, img_size))\n            except Exception:\n                ok = False\n                break\n        if ok:\n            clips.append(np.stack(chan, axis=0))  # (3, H, W)\n    return clips\n\n\nif HAS_TORCH:\n    # ---------------- 数据集(v2:按平面过滤 series) ----------------\n\n    class KneeDataset(Dataset):\n        """study 级数据集,只取指定 Anatomical_Plane 的 series,拼成 (C*3, H, W)。"""\n\n        def __init__(self, studies, labels_df, dcm_root, series_df, plane,\n                     has_targets=True, n_clips=2, sampling_mode="legacy"):\n            self.studies = studies\n            self.labels_df = labels_df\n            self.dcm_root = dcm_root\n            self.series_df = series_df\n            self.plane = plane\n            self.has_targets = has_targets\n            self.n_clips = n_clips\n            self.sampling_mode = sampling_mode\n            self.cache = {}  # 每折独立缓存:fold 结束随 dataset 释放,避免 5 折累积 OOM\n            # 诊断计数:读到 clip 的 study / 空 study / 累计 clip 数\n            self.n_hit = 0\n            self.n_empty = 0\n            self.n_clips_total = 0\n\n        def stats(self):\n            """返回 (读到 clip 的 study 数, 空 study 数, 平均 clip 数),防止静默白训练。"""\n            total = self.n_hit + self.n_empty\n            avg = (self.n_clips_total / self.n_hit) if self.n_hit else 0.0\n            return self.n_hit, self.n_empty, avg\n\n        def _load_study(self, uid):\n            key = (uid, self.plane)\n            if key in self.cache:\n                return self.cache[key]\n            srows = self.series_df[self.series_df["StudyInstanceUID"] == uid]\n            if "Anatomical_Plane" in srows.columns:\n                srows = srows[srows["Anatomical_Plane"] == self.plane]\n                srows = srows.sort_values("Fluid_Sensitive", kind="stable").drop_duplicates(\n                    subset=["Fluid_Sensitive"]).head(MAX_SERIES)\n            else:\n                srows = srows.head(MAX_SERIES)\n            clips = []\n            for _, sr in srows.iterrows():\n                sid = sr["SeriesInstanceUID"]\n                rows = read_dicom_slices(\n                    self.dcm_root, sid, study_uid=uid, sampling_mode=self.sampling_mode\n                )\n                clips.extend(\n                    make_clips(rows, n_clips=self.n_clips, sampling_mode=self.sampling_mode)\n                )\n            self.cache[uid] = clips\n            # 仅首次加载计数(缓存命中不再累加,避免多 epoch 重复)\n            if clips:\n                self.n_hit += 1\n                self.n_clips_total += len(clips)\n            else:\n                self.n_empty += 1\n            return clips\n\n        def __len__(self):\n            return len(self.studies)\n\n        def __getitem__(self, i):\n            uid = self.studies[i]\n            clips = self._load_study(uid)  # 命中/空计数在 _load_study 首次加载时完成\n            in_ch = self.n_clips * CLIP  # 输入通道数:clip 数 × 每 clip 3 切片\n            if not clips:\n                x = np.zeros((in_ch, IMG_SIZE, IMG_SIZE), dtype=np.float32)\n            else:\n                x = np.concatenate(clips[: self.n_clips], axis=0)  # (k*3, H, W)\n                pad = in_ch - x.shape[0]\n                if pad > 0:\n                    x = np.concatenate([x, np.zeros((pad, IMG_SIZE, IMG_SIZE), dtype=np.float32)])\n                elif pad < 0:\n                    x = x[: in_ch]\n            x = torch.tensor(x, dtype=torch.float32)\n            y = torch.zeros(len(LABELS), dtype=torch.float32)\n            valid = torch.zeros(len(LABELS), dtype=torch.float32)\n            if self.has_targets:\n                row = self.labels_df[self.labels_df["StudyInstanceUID"] == uid]\n                if len(row):\n                    for j, c in enumerate(LABELS):\n                        v = row.iloc[0][c]\n                        if pd.notna(v):\n                            y[j] = float(v)\n                            valid[j] = 1.0\n            return x, y, valid\n\n\n    # ---------------- 模型 ----------------\n\n    _PRETRAINED_TRIED = [False]  # 模块级标志:下载失败过一次就不再反复重试\n\n\n    def _build_resnet34(use_pretrained=False, offline_paths=None):\n        """构造 ResNet34 并加载 ImageNet 预训练权重。\n\n        权重来源优先级:\n        1. offline_paths:挂载的离线权重文件(扫描 /kaggle/input 得到,无需联网);\n        2. use_pretrained=True:联网下载 torchvision 权重(带重试,失败回退随机);\n        3. 否则随机初始化。\n        成功加载后权重由 torch 缓存,后续 fold 秒加载。\n        """\n        net = tvm.resnet34(weights=None)\n        if offline_paths:\n            ok, desc = _try_load_offline(net, offline_paths)\n            if ok:\n                return net\n            print(f"[权重] 离线候选均不匹配({desc}),回退到随机/联网路径", flush=True)\n        if not use_pretrained:\n            print("[权重] 未开启 --pretrained 且无离线权重,使用随机初始化"\n                  "(不联网;如需预训练请加 --pretrained 并开启 Internet,"\n                  "或挂载离线权重数据集)", flush=True)\n            return net\n        if _PRETRAINED_TRIED[0]:\n            return net  # 之前已确认下载不了,直接随机\n        for attempt in range(1, 4):\n            try:\n                net = tvm.resnet34(weights=tvm.ResNet34_Weights.IMAGENET1K_V1\n                                   if hasattr(tvm, "ResNet34_Weights") else None)\n                return net\n            except Exception as e:\n                print(f"[权重] ResNet34 预训练下载失败(第 {attempt}/3 次): {e}", flush=True)\n                time.sleep(5)\n        _PRETRAINED_TRIED[0] = True\n        print("[权重] 3 次均失败,改用随机初始化(精度会明显下降;建议重试或挂载预训练权重数据集)", flush=True)\n        return tvm.resnet34(weights=None)\n\n\n    class KneeModel(nn.Module):\n        """ResNet34 共享 backbone + 全局池化 + 12 sigmoid 头(每个平面一个实例)。"""\n\n        def __init__(self, num_labels=len(LABELS), channels=CLIP, pretrained=False, offline_paths=None):\n            super().__init__()\n            if HAS_TV:\n                self.backbone = _build_resnet34(use_pretrained=pretrained,\n                                                offline_paths=offline_paths)\n            else:\n                raise RuntimeError("需要 torchvision(请在 Kaggle Notebook GPU 环境运行)")\n            conv1_w = self.backbone.conv1.weight  # 保留预训练 conv1 权重\n            self.backbone.conv1 = nn.Conv2d(channels, 64, kernel_size=7, stride=2, padding=3, bias=False)\n            if conv1_w.shape == self.backbone.conv1.weight.shape:  # channels==3 时直接沿用\n                self.backbone.conv1.weight.data.copy_(conv1_w.data)\n            elif channels % 3 == 0 and conv1_w.shape[1] == 3:\n                # 2.5D 通道扩展:把 3 通道预训练权重复制 n_clips 份(常见做法)\n                rep = channels // 3\n                self.backbone.conv1.weight.data.copy_(conv1_w.repeat(1, rep, 1, 1))\n                print(f"[权重] conv1 预训练权重复制 {rep} 份扩展到 {channels} 通道", flush=True)\n            self.backbone.fc = nn.Identity()\n            self.head = nn.Linear(512, num_labels)\n\n        def forward(self, x):\n            feat = self.backbone(x)  # (B, 512)\n            return self.head(feat)   # logits (B, 12)\n\n\n\n# ---------------- 训练 / 验证 ----------------\n\ndef train_epoch(model, loader, opt, crit, device):\n    model.train()\n    tot, cnt = 0.0, 0\n    for x, y, valid in loader:\n        x, y, valid = x.to(device), y.to(device), valid.to(device)\n        opt.zero_grad()\n        out = model(x)\n        if valid.sum() == 0:\n            continue\n        # EXP-013: confidence-weight soft BCE — w = 2*|y-0.5| (UNK/silence already NaN)\n        w = 2.0 * (y - 0.5).abs()\n        denom = (valid * w).sum()\n        loss = (crit(out, y) * valid * w).sum() / denom.clamp(min=1e-6)\n        loss.backward()\n        opt.step()\n        tot += loss.item() * len(x)\n        cnt += len(x)\n    return tot / max(cnt, 1)\n\n\ndef predict(model, loader, device):\n    model.eval()\n    rows = []\n    with torch.no_grad():\n        for x, _, _ in loader:\n            x = x.to(device)\n            out = torch.sigmoid(model(x))\n            # .cpu().tolist() 而非 .cpu().numpy():torch 2.2.0+cu118 编于 numpy1.x,\n            # 在 Kaggle numpy2.0.2 下 torch→numpy interop 崩;tolist 纯 C++ 路径安全。\n            rows.extend(out.cpu().tolist())\n    return np.asarray(rows, dtype=np.float32) if rows else np.zeros((0, len(LABELS)), dtype=np.float32)\n\n\ndef run_fold(model, train_ds, val_ds, device, args):\n    train_loader = DataLoader(train_ds, batch_size=args.batch, shuffle=True,\n                              num_workers=args.workers, drop_last=False)\n    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False,\n                            num_workers=args.workers, drop_last=False)\n    opt = AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    sched = CosineAnnealingLR(opt, T_max=args.epochs)\n    crit = nn.BCEWithLogitsLoss(reduction="none")\n    for ep in range(args.epochs):\n        t0 = time.time()\n        loss = train_epoch(model, train_loader, opt, crit, device)\n        sched.step()\n        if args.verbose:\n            print(f"  epoch {ep + 1}/{args.epochs} loss {loss:.4f} ({time.time() - t0:.0f}s)", flush=True)\n    val_pred = predict(model, val_loader, device)\n    # DICOM 命中诊断:训练+验证已迭代完毕,此时计数才准确\n    th, eh, at = train_ds.stats()\n    vh, ev, av = val_ds.stats()\n    print(f"[数据] train 命中 {th}/{len(train_ds)} study(空 {eh}, 平均 {at:.1f} clip) | "\n          f"val 命中 {vh}/{len(val_ds)} study(空 {ev})", flush=True)\n    if th == 0 or vh == 0:\n        print("[警告] 有折 DICOM 全部没读到(命中 0)——输入将是全零,分数必然 ~0.5!", flush=True)\n    val_uid = np.array(val_ds.studies)  # shuffle=False,顺序与数据集一致\n    return val_pred, val_uid\n\n\n# ---------------- 推理 / 融合 / 提交 ----------------\n\ndef predict_test(model, test_df, series_df, root, device, plane, n_clips=2, sampling_mode="legacy"):\n    """对 test 每个 study 按指定平面收集 clips,预测 12 个概率。"""\n    ds = KneeDataset(test_df["StudyInstanceUID"].tolist(), None, root, series_df,\n                     plane, has_targets=False, n_clips=n_clips, sampling_mode=sampling_mode)\n    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=0)\n    return predict(model, loader, device)\n\n\ndef plane_oof_auc(oof, labels_df, uid_list):\n    """计算一个平面模型在指定 study 子集上的宏平均 OOF AUC(只算有标注的列)。\n\n    labels_df 必须是【官方 gold 原始表】(0/1/NaN,如 train_csv 原列),不能传\n    load_labels 填充后的 soft 表——soft 伪标签既非 ground truth,也会让\n    roc_auc_score 崩(continuous)。58 例小样本下某列可能只有单一类别,跳过该列。\n    """\n    aucs = []\n    for j, c in enumerate(LABELS):\n        y_true = labels_df.set_index("StudyInstanceUID")[c].reindex(uid_list).values.astype(float)\n        valid = ~np.isnan(y_true)\n        if valid.sum() >= 20:\n            yv = y_true[valid]\n            if len(np.unique(yv)) < 2:\n                continue\n            aucs.append(roc_auc_score(yv, oof[valid, j]))\n    return float(np.mean(aucs)) if aucs else 0.0\n\n\n# ---------------- artifact / resume / ensemble helpers ----------------\n\ndef _artifact_root(args):\n    base = Path(getattr(args, "artifact_dir", "") or "artifacts")\n    exp = getattr(args, "experiment_id", "exp_001_5fold_sagittal")\n    return base / exp\n\n\ndef _fold_meta_path(artifact_root, fold):\n    return artifact_root / f"fold{fold}" / "meta.json"\n\n\ndef _is_fold_done(artifact_root, fold):\n    p = _fold_meta_path(artifact_root, fold)\n    if not p.exists():\n        return False\n    try:\n        meta = json.loads(p.read_text(encoding="utf-8"))\n        return meta.get("completion_status") == "done"\n    except Exception:\n        return False\n\n\ndef _save_fold_artifact(artifact_root, fold, test_pred, test_uids, meta):\n    fold_dir = artifact_root / f"fold{fold}"\n    fold_dir.mkdir(parents=True, exist_ok=True)\n    np.save(str(fold_dir / "test_pred.npy"), test_pred.astype(np.float32))\n    pd.DataFrame({"StudyInstanceUID": test_uids}).to_csv(\n        fold_dir / "test_uid.csv", index=False\n    )\n    meta["completion_status"] = "done"\n    meta["prediction_shape"] = list(test_pred.shape)\n    (fold_dir / "meta.json").write_text(\n        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n\n\ndef _load_fold_predictions(artifact_root, fold, canonical_uids):\n    fold_dir = artifact_root / f"fold{fold}"\n    pred = np.load(fold_dir / "test_pred.npy")\n    uids = pd.read_csv(fold_dir / "test_uid.csv")["StudyInstanceUID"].tolist()\n    if pred.shape != (len(canonical_uids), len(LABELS)):\n        raise ValueError(f"fold{fold} shape mismatch: {pred.shape} vs expected {(len(canonical_uids), len(LABELS))}")\n    if uids != canonical_uids:\n        raise ValueError(f"fold{fold} UID order mismatch")\n    if not np.isfinite(pred).all():\n        raise ValueError(f"fold{fold} contains NaN/Inf")\n    if pred.min() < 0 or pred.max() > 1:\n        raise ValueError(f"fold{fold} predictions out of [0,1]")\n    return pred\n\n\ndef _save_experiment_meta(artifact_root, train_fold_uids, test_uids, args):\n    meta = {\n        "experiment_id": getattr(args, "experiment_id", "exp_001_5fold_sagittal"),\n        "git_commit": __git_commit(),\n        "args": vars(args),\n        "labels": LABELS,\n        "train_fold0_uids": train_fold_uids,\n        "test_uids": test_uids,\n    }\n    (artifact_root / "config.json").write_text(\n        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    pd.DataFrame({"StudyInstanceUID": test_uids}).to_csv(\n        artifact_root / "test_uid.csv", index=False\n    )\n    fold_rows = []\n    for k in range(args.folds):\n        fold_rows.append({"fold": k, "train_uids": train_fold_uids[k]})\n    pd.DataFrame(fold_rows).to_csv(artifact_root / "fold_assignment.csv", index=False)\n\n\ndef _maybe_kaggle_dataset_version(artifact_root, message):\n    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") != "Interactive":\n        return\n    slug = os.environ.get("KAGGLE_KERNEL_SLUG", "rsna-artifacts-tmp")\n    owner = os.environ.get("KAGGLE_USERNAME", "ziyiming")\n    dataset_id = f"{owner}/{slug}"\n    try:\n        subprocess.run(\n            [\n                "kaggle",\n                "datasets",\n                "version",\n                "-p",\n                str(artifact_root),\n                "-m",\n                message,\n                "--dir-mode",\n                "tar",\n            ],\n            check=True,\n            capture_output=True,\n        )\n        print(f"[artifact] uploaded dataset version: {dataset_id}", flush=True)\n    except Exception as e:\n        print(f"[artifact] dataset upload skipped: {e}", flush=True)\n\n\ndef __git_commit():\n    try:\n        import subprocess\n        return subprocess.check_output(\n            ["git", "rev-parse", "HEAD"], text=True\n        ).strip()\n    except Exception:\n        return "unknown"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--epochs", type=int, default=2)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--lr", type=float, default=3e-4)\n    ap.add_argument("--workers", type=int, default=0)\n    ap.add_argument("--n-clips", type=int, default=2)\n    ap.add_argument("--sampling-mode", type=str, default="legacy",\n                    choices=["legacy", "physical"],\n                    help="EXP-016: legacy=z-sort+index clips; physical=IOP normal sort + physical 2.5D")\n    ap.add_argument("--seed", type=int, default=SEED)\n    ap.add_argument("--folds", type=int, default=FOLDS)\n    ap.add_argument("--only-fold", type=int, default=-1,\n                    help="只跑指定折(其余跳过;默认 -1=全部)。探路用:--folds 5 --only-fold 0 训第 0 折"\n                         "(train 4/5 与正式同分布),test 推理用该折模型出提交")\n    ap.add_argument("--planes", type=str, default=",".join(PLANES),\n                    help="逗号分隔的平面子集,如 axial,sagittal(小写,不区分大小写)")\n    ap.add_argument("--fuse", type=str, default="weight",\n                    choices=["weight", "mean"],\n                    help="平面融合方式:weight=按 OOF AUC 加权;mean=简单平均")\n    ap.add_argument("--pseudo-conf", type=float, default=0.0,\n                    help="伪标签置信度过滤阈值(0=不过滤;0.5=丢弃置信度<0.5的伪标签)")\n    ap.add_argument("--no-report-labels", action="store_true",\n                    help="禁用内置报告关键词软标签兜底(默认开启:未挂社区数据集时"\n                         "自动从 4407 份放射报告生成软标签,零挂载)")\n    ap.add_argument("--verbose", action="store_true")\n    ap.add_argument("--pretrained", action="store_true",\n                    help="尝试下载 ImageNet 预训练权重(需开启 Internet);默认随机初始化不联网")\n    ap.add_argument("--offline-weights", type=str, default="",\n                    help="逗号分隔的离线权重 .pth 路径(优先级最高);留空则自动扫描挂载目录")\n    ap.add_argument("--no-offline-scan", action="store_true",\n                    help="禁用自动扫描 /kaggle/input 下的离线权重文件")\n    ap.add_argument("--data", type=str, default="",\n                    help="显式指定数据根目录(如 kagglehub.competition_download 返回的 path),"\n                         "指定后跳过自动查找")\n    ap.add_argument("--artifact-dir", type=str, default="artifacts",\n                    help="artifact 根目录(默认 artifacts)")\n    ap.add_argument("--experiment-id", type=str, default="exp_001_5fold_sagittal",\n                    help="experiment ID(用于 artifact 子目录)")\n    ap.add_argument("--resume-dir", type=str, default="",\n                    help="已有 artifact 目录,启动时扫描 meta.json 跳过已完成 fold")\n    args, _ = ap.parse_known_args()\n\n    resume_root = Path(args.resume_dir) if args.resume_dir else None\n    if resume_root and args.artifact_dir and not args.experiment_id:\n        args.experiment_id = resume_root.name\n\n    if not HAS_TORCH or not HAS_TV or not HAS_PYDICOM:\n        sys.exit("依赖缺失:请在 Kaggle Notebook(GPU)环境运行(torch/torchvision/pydicom);本地仅可做数据层验证。")\n\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n\n    if args.data:\n        if os.path.exists(os.path.join(args.data, "train.csv")):\n            root, out = args.data, args.data\n            print(f"[数据] 使用 --data 指定目录: {root}")\n        else:\n            sys.exit(f"--data 指定目录 {args.data} 下没有 train.csv,请检查路径")\n    else:\n        root, out = resolve_paths()\n    print(f"[数据] root={root}  out={out}")\n\n    train_csv = pd.read_csv(os.path.join(root, "train.csv"))\n    labels_df = load_labels(root, pseudo_conf=args.pseudo_conf,\n                            use_report_labels=not args.no_report_labels)\n    series_df = pd.read_csv(os.path.join(root, "train_series.csv"))\n    test_df = pd.read_csv(os.path.join(root, "test.csv"))\n    test_series = pd.read_csv(os.path.join(root, "test_series.csv"))\n    dcm_train = os.path.join(root, "train_series")\n    dcm_test = os.path.join(root, "test_series")\n    print(f"[数据] train {len(train_csv)} study / test {len(test_df)} study / series {len(series_df)}")\n    _print_dcm_tree(root, "DICOM")  # 打印 train_series 真实目录结构,确认布局\n\n    print(f"[采样] sampling_mode={args.sampling_mode} n_clips={args.n_clips} "\n          f"HAS_SG={HAS_SG}")\n    planes = [p.strip().title() for p in args.planes.split(",") if p.strip()]\n    planes = [p for p in planes if p in PLANES]\n    if not planes:\n        sys.exit(f"--planes 无效,可选: {PLANES}")\n    print(f"[平面] 使用 {planes}")\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"[设备] {device}")\n\n    # 离线预训练权重:显式指定优先,否则自动扫描 /kaggle/input\n    offline_paths = []\n    if args.offline_weights.strip():\n        for p in [x.strip() for x in args.offline_weights.split(",") if x.strip()]:\n            if os.path.isfile(p):\n                offline_paths.append((p, p))\n            else:\n                print(f"[权重] --offline-weights 指定的文件不存在,跳过: {p}", flush=True)\n    elif not args.no_offline_scan:\n        offline_paths = _scan_offline_weights()\n        if offline_paths:\n            print(f"[权重] 自动扫描到 {len(offline_paths)} 个离线权重候选:", flush=True)\n            for p, desc in offline_paths[:6]:\n                print(f"  {desc}", flush=True)\n            if len(offline_paths) > 6:\n                print(f"  ... 共 {len(offline_paths)} 个", flush=True)\n        else:\n            print("[权重] 未在挂载目录发现离线权重(可挂载社区打包的 resnet34 权重数据集)", flush=True)\n            _print_mount_tree("/kaggle/input", "权重")\n\n    gkf = GroupKFold(n_splits=args.folds)\n    groups = train_csv["StudyInstanceUID"].values\n\n    # 每个平面独立跑折,记录 OOF 与 test 预测\n    plane_oof, plane_test, plane_auc, plane_uids = {}, {}, {}, {}\n    for plane in planes:\n        print(f"\\n########## 平面: {plane} ##########")\n        oof = np.zeros((len(train_csv), len(LABELS)))\n        artifact_root = _artifact_root(args)\n        canonical_test_uids = test_df["StudyInstanceUID"].tolist()\n        train_fold_uids = []\n        for k, (tr_idx, va_idx) in enumerate(gkf.split(train_csv, groups=groups)):\n            tr_studies = train_csv.iloc[tr_idx]["StudyInstanceUID"].tolist()\n            va_studies = train_csv.iloc[va_idx]["StudyInstanceUID"].tolist()\n            train_fold_uids.append(tr_studies)\n            if args.only_fold >= 0 and k != args.only_fold:\n                continue\n            if resume_root and _is_fold_done(artifact_root, k):\n                print(f"[artifact] fold {k} done, skip", flush=True)\n                continue\n            tr_ds = KneeDataset(tr_studies, labels_df, dcm_train, series_df, plane,\n                                n_clips=args.n_clips, sampling_mode=args.sampling_mode)\n            va_ds = KneeDataset(va_studies, labels_df, dcm_train, series_df, plane,\n                                n_clips=args.n_clips, sampling_mode=args.sampling_mode)\n            model = KneeModel(pretrained=args.pretrained,\n                              offline_paths=offline_paths,\n                              channels=args.n_clips * CLIP).to(device)\n            print(f"[折 {k + 1}/{args.folds}] {plane} train {len(tr_ds)} / val {len(va_ds)}", flush=True)\n            t0 = time.time()\n            val_pred, val_uid = run_fold(model, tr_ds, va_ds, device, args)\n            fold_train_time = time.time() - t0\n            for j, uid in enumerate(val_uid):\n                row = np.where(train_csv["StudyInstanceUID"].values == uid)[0]\n                if len(row):\n                    oof[row[0]] = val_pred[j]\n            t0 = time.time()\n            test_pred = predict_test(model, test_df, test_series, dcm_test, device, plane,\n                                     n_clips=args.n_clips, sampling_mode=args.sampling_mode)\n            fold_inference_time = time.time() - t0\n            meta = {\n                "fold": k,\n                "plane": plane,\n                "n_clips": args.n_clips,\n                "sampling_mode": getattr(args, "sampling_mode", "legacy"),\n                "image_size": IMG_SIZE,\n                "epochs": args.epochs,\n                "batch_size": args.batch,\n                "seed": args.seed,\n                "git_commit": __git_commit(),\n                "fold_train_time_sec": fold_train_time,\n                "fold_inference_time_sec": fold_inference_time,\n            }\n            _save_fold_artifact(artifact_root, k, test_pred, canonical_test_uids, meta)\n            try:\n                wdir = artifact_root / f"fold{k}" / plane\n                wdir.mkdir(parents=True, exist_ok=True)\n                wpath = wdir / "model.pt"\n                torch.save(model.state_dict(), str(wpath))\n                meta2 = dict(meta)\n                meta2["channels"] = int(args.n_clips * CLIP)\n                meta2["num_labels"] = len(LABELS)\n                (wdir / "model_meta.json").write_text(\n                    json.dumps(meta2, ensure_ascii=False, indent=2), encoding="utf-8")\n                print(f"[weights] saved {wpath}", flush=True)\n            except Exception as e:\n                print(f"[weights] save failed: {e}", flush=True)\n            print(f"[artifact] saved fold{k} -> {artifact_root / f\'fold{k}\'}", flush=True)\n        # experiment-level meta once per plane\n        _save_experiment_meta(artifact_root, train_fold_uids, canonical_test_uids, args)\n        print(f"[artifact] saved experiment meta -> {artifact_root}", flush=True)\n        # 该平面 OOF AUC(宏平均)\n        uid_list = train_csv["StudyInstanceUID"].tolist()\n        if args.only_fold >= 0:\n            print(f"[{plane}] 探路模式(--only-fold {args.only_fold}):OOF 单折不完整,"\n                  f"跳过 OOF AUC(权重记 0=均等);test 推理用该折模型")\n            auc = 0.0\n        else:\n            auc = plane_oof_auc(oof, train_csv[["StudyInstanceUID"] + LABELS], uid_list)\n            print(f"[{plane}] OOF 宏平均 AUC {auc:.4f}")\n        plane_oof[plane], plane_auc[plane], plane_uids[plane] = oof, auc, uid_list\n        # load saved fold predictions for this plane\n        per_fold = []\n        missing = []\n        for k in range(args.folds):\n            p = artifact_root / f"fold{k}" / "test_pred.npy"\n            if not p.exists():\n                missing.append(k)\n                continue\n            try:\n                per_fold.append(_load_fold_predictions(artifact_root, k, canonical_test_uids))\n            except Exception as e:\n                missing.append(k)\n                print(f"[ensemble] fold{k} invalid: {e}", flush=True)\n        if len(missing) == args.folds:\n            sys.exit(f"[ensemble] no valid fold predictions available under {artifact_root}")\n        if missing:\n            print(f"[ensemble] missing folds: {missing}; ensemble with available folds only", flush=True)\n        plane_test[plane] = np.mean(per_fold, axis=0) if per_fold else None\n        if plane_test[plane] is not None:\n            print(f"[ensemble] {plane} ensembled from {len(per_fold)} folds, shape={plane_test[plane].shape}")\n\n    # 融合\n    if args.fuse == "weight":\n        wsum = sum(max(a, 1e-6) for a in plane_auc.values())\n        weights = {p: max(a, 1e-6) / wsum for p, a in plane_auc.items()}\n        print(f"[融合] OOF AUC 加权: { {p: round(w, 4) for p, w in weights.items()} }")\n    else:\n        weights = {p: 1.0 / len(planes) for p in planes}\n        print("[融合] 简单平均")\n    fused = np.zeros_like(plane_test[planes[0]])\n    for p, w in weights.items():\n        fused += w * plane_test[p]\n    sub = pd.DataFrame({"StudyInstanceUID": test_df["StudyInstanceUID"], **{\n        LABELS[j]: fused[:, j] for j in range(len(LABELS))}})\n    sub_path = os.path.join(out, "submission.csv")\n    sub.to_csv(sub_path, index=False)\n    print(f"\\n提交文件已生成: {sub_path}  shape={sub.shape}")\n\n\nif __name__ == "__main__":\n    main()\n')
print('embedded', os.path.getsize('sampling_geometry.py'), os.path.getsize('rsna_v3.py'))


In [ ]:
import os, sys, random, json
import numpy as np
import pandas as pd
import pydicom
sys.path.insert(0, os.getcwd())
import sampling_geometry as SG

def find_root(base='/kaggle/input', depth=0, max_depth=3):
    if not os.path.isdir(base):
        return None
    try:
        entries = sorted(os.listdir(base))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(base, e)
        if not os.path.isdir(p) or e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'train.csv')) and os.path.isdir(os.path.join(p, 'train_series')):
            return p
        if depth < max_depth - 1:
            r = find_root(p, depth + 1, max_depth)
            if r:
                return r
    return None

ROOT = find_root()
print('ROOT', ROOT)
assert ROOT
series = pd.read_csv(os.path.join(ROOT, 'train_series.csv'))
train_series_dir = os.path.join(ROOT, 'train_series')

def files_for(uid, sid):
    for c in [os.path.join(train_series_dir, uid, sid), os.path.join(train_series_dir, sid)]:
        if os.path.isdir(c):
            return [os.path.join(c, f) for f in os.listdir(c) if f.lower().endswith('.dcm')]
    return []

def hdr(path):
    return pydicom.dcmread(path, stop_before_pixels=True)

random.seed(42)
uids = series['StudyInstanceUID'].drop_duplicates().tolist()
sample_uids = random.sample(uids, 40)
SG.reset_stats()
rows = []
sort_diff_series = 0
n_series = 0
for uid in sample_uids:
    sub = series[series.StudyInstanceUID == uid]
    for _, sr in sub.iterrows():
        sid = sr.SeriesInstanceUID
        files = files_for(uid, sid)
        if len(files) < 3:
            continue
        n_series += 1
        legacy = []
        for f in files:
            try:
                d = hdr(f)
                ipp = d.get('ImagePositionPatient')
                z = float(ipp[2]) if ipp is not None and len(ipp) >= 3 else None
            except Exception:
                z = None
            legacy.append((f, z))
        legacy.sort(key=lambda r: (r[1] is None, r[1] if r[1] is not None else 0.0))
        phys, meta = SG.sort_slices_physical(files, hdr)
        leg_order = [r[0] for r in legacy]
        phy_order = [r[0] for r in phys]
        disagree = float(np.mean([a != b for a, b in zip(leg_order, phy_order)])) if len(leg_order) == len(phy_order) else 1.0
        if disagree > 0:
            sort_diff_series += 1
        clips, metas = SG.make_clips_physical(
            phys, n_clips=4, img_size=32,
            window_fn=lambda d, s=32: np.zeros((s, s), np.float32),
            dcm_read_fn=hdr,
        )
        gaps = []
        for m in metas:
            for x in (m.get('gaps') or (None, None)):
                if x is not None and np.isfinite(x):
                    gaps.append(float(x))
        pos = [p for _, p, _k in phys]
        spacing = SG.median_spacing(pos)
        pv = [p for p in pos if p is not None and np.isfinite(p)]
        span = float(max(pv) - min(pv)) if len(pv) >= 2 else np.nan
        rows.append(dict(
            uid=uid, sid=sid, plane=str(sr.Anatomical_Plane), n_files=len(files),
            sort_disagree=disagree, n_clips_made=len(clips),
            relaxed=sum(1 for m in metas if m.get('relaxed')),
            gap_med=float(np.median(gaps)) if gaps else np.nan,
            gap_p90=float(np.percentile(gaps, 90)) if gaps else np.nan,
            gap_max=float(np.max(gaps)) if gaps else np.nan,
            spacing=spacing, span=span,
            meta_phys=meta['n_physical'], meta_z=meta['n_ipp_z'], meta_miss=meta['n_missing'],
        ))

df = pd.DataFrame(rows)
print(df.head())
print('series tested', n_series, 'sort_disagree>0', sort_diff_series, f'({sort_diff_series/max(n_series,1):.1%})')
print('SG.STATS', SG.STATS)
den = max(1, SG.STATS['sort_physical_ok'] + SG.STATS['sort_ipp_z_fallback'] + SG.STATS['sort_no_ipp'])
print('fallback_rate', (SG.STATS['sort_ipp_z_fallback'] + SG.STATS['sort_no_ipp']) / den)
print('median adjacent gap', float(np.nanmedian(df.gap_med)))
print('P90 adjacent gap median', float(np.nanmedian(df.gap_p90)) if df.gap_p90.notna().any() else None)
print('P90 of series gap_p90', float(np.nanpercentile(df.gap_p90.dropna(), 90)) if df.gap_p90.notna().any() else None)
print('clips/series median', float(df.n_clips_made.median()))
print('span median', float(np.nanmedian(df.span)))
print('relaxed_rate', float(df.relaxed.sum() / max(df.n_triplet if 'n_triplet' in df else df.n_clips_made.sum(), 1)) if df.n_clips_made.sum() else None)
df.to_csv('exp016_smoke_metrics.csv', index=False)
print('wrote exp016_smoke_metrics.csv')


In [ ]:
import re, sys, os
sys.path.insert(0, os.getcwd())
src = open('rsna_v3.py', encoding='utf-8').read()
assert 'sampling-mode' in src or 'sampling_mode' in src
assert 'physical' in src and 'legacy' in src
try:
    import rsna_v3 as V
    print('LABELS', V.LABELS)
    print('order_ok', V.LABELS[0] == 'ACL' and V.LABELS[-1] == "Fracture")
except Exception as e:
    print('rsna_v3 import note', type(e), e)
    expected = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
    print('expected LABELS', expected)
for p in [os.path.join(os.path.dirname(ROOT), 'sample_submission.csv'),
          '/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv']:
    if os.path.isfile(p):
        print('submission cols', list(pd.read_csv(p, nrows=0).columns))
        break
print('UID order: KneeDataset consumes studies list unchanged')
print('SMOKE_DONE')
